In [ ]:
import os
import pandas as pd
import pprint

In [ ]:
label_counts = {
    "academia_pseudocode": 0,
    "industry_pseudocode": 0,
    "academia_open_source_code": 0,
    "industry_open_source_code": 0,    
    "academia_open_datasets": 0,
    "industry_open_datasets": 0,
    "academia_dataset_splits": 0,
    "industry_dataset_splits": 0,
    "academia_hardware_specification": 0,
    "industry_hardware_specification": 0,
    "academia_software_dependencies": 0,
    "industry_software_dependencies": 0,
    "academia_experiment_setup": 0,
    "industry_experiment_setup": 0,
    "academia_total": 0,
    "industry_total": 0,
    "total": 0
}


In [ ]:
exclude_theoretical = True

# Open llm_results.csv
csv_file_path = "../../results/llm_results.csv"

if not os.path.exists(csv_file_path):
    raise FileNotFoundError(f"CSV file not found: {csv_file_path}")

df = pd.read_csv(csv_file_path)

In [ ]:
len(df)

In [ ]:
if exclude_theoretical:
    # remove all of the rows where research_type is 1
    df = df[df["research_type"] != 1]

In [ ]:
len(df)

In [ ]:
label_results = {}

conferences = ["AAAI", "ICML", "ICLR", "IJCAI", "NeurIPS"]
years = list(range(2014, 2025))
             
for conference in conferences:
    label_results[conference] = {}
    for year in years:
        if conference == "IJCAI" and year == 2014:
            continue
        label_results[conference][year] = label_counts.copy()

print(label_results)

In [ ]:
df.head()

In [ ]:

for _, row in df.iterrows():
    year = row["year"]
    conference = row["conference"]

    if row["affiliation"] == 0:
        if row["pseudocode"] == 1:
            label_results[conference][year]["academia_pseudocode"] += 1
        if row["open_source_code"] == 1:
            label_results[conference][year]["academia_open_source_code"] += 1
        if row["train"] == 1:
            label_results[conference][year]["academia_open_datasets"] += 1
        if row["validation"] == 1:
            label_results[conference][year]["academia_dataset_splits"] += 1
        if row["hardware_specification"] == 1:
            label_results[conference][year]["academia_hardware_specification"] += 1
        if row["software_dependencies"] == 1:
            label_results[conference][year]["academia_software_dependencies"] += 1
        if row["experiment_setup"] == 1:
            label_results[conference][year]["academia_experiment_setup"] += 1

        label_results[conference][year]["academia_total"] += 1

    elif row["affiliation"] == 1 or row["affiliation"] == 2:
        if row["pseudocode"] == 1:
            label_results[conference][year]["industry_pseudocode"] += 1
        if row["open_source_code"] == 1:
            label_results[conference][year]["industry_open_source_code"] += 1
        if row["train"] == 1:
            label_results[conference][year]["industry_open_datasets"] += 1
        if row["validation"] == 1:
            label_results[conference][year]["industry_dataset_splits"] += 1
        if row["hardware_specification"] == 1:
            label_results[conference][year]["industry_hardware_specification"] += 1
        if row["software_dependencies"] == 1:
            label_results[conference][year]["industry_software_dependencies"] += 1
        if row["experiment_setup"] == 1:
            label_results[conference][year]["industry_experiment_setup"] += 1

        label_results[conference][year]["industry_total"] += 1

    label_results[conference][year]["total"] += 1


In [ ]:
pprint.pprint(label_results["IJCAI"][2016])

In [ ]:
for conference in label_results:

    for year in label_results[conference]:
        for key in label_results[conference][year]:
            if key not in ["academia_total", "industry_total", "total"]:
                if label_results[conference][year]["academia_total"] > 0:
                    if "academia" in key:
                        label_results[conference][year][key] = label_results[conference][year][key] / label_results[conference][year]["academia_total"]
                if label_results[conference][year]["industry_total"] > 0:
                    if "industry" in key:
                        label_results[conference][year][key] = label_results[conference][year][key] / label_results[conference][year]["industry_total"]
                if label_results[conference][year]["total"] > 0:
                    if "total" in key:
                        label_results[conference][year][key] = label_results[conference][year][key] / label_results[conference][year]["total"]
        # if the "total" in key, remove the key
        label_results[conference][year].pop("total", None)
        label_results[conference][year].pop("academia_total", None)
        label_results[conference][year].pop("industry_total", None)


In [ ]:
academia_greater_count = 0
industry_greater_count = 0
tie_count = 0

academia_greater_by_year = {}
for year in years:
    academia_greater_by_year[year] = 0

industry_greater_by_year = {}
for year in years:
    industry_greater_by_year[year] = 0

for conference in label_results:
    for year in label_results[conference]:

        # compare academia and industry pseudocode
        if label_results[conference][year]["academia_pseudocode"] > label_results[conference][year]["industry_pseudocode"]:
            #print(f"In {conference} {year}, academia had more pseudocode ({label_results[conference][year]['academia_pseudocode']:.2f}) than industry ({label_results[conference][year]['industry_pseudocode']:.2f})")
            academia_greater_count += 1
            academia_greater_by_year[year] += 1
        elif label_results[conference][year]["academia_pseudocode"] < label_results[conference][year]["industry_pseudocode"]:
            #print(f"In {conference} {year}, industry had more pseudocode ({label_results[conference][year]['industry_pseudocode']:.2f}) than academia ({label_results[conference][year]['academia_pseudocode']:.2f})")
            industry_greater_count += 1
            industry_greater_by_year[year] += 1
        else:
            print(f"In {conference} {year}, pseudocode was tied ({label_results[conference][year]['academia_pseudocode']:.2f})")
            tie_count += 1
        # compare academia and industry open source code

        if label_results[conference][year]["academia_open_source_code"] > label_results[conference][year]["industry_open_source_code"]:
            #print(f"In {conference} {year}, academia had more open source code ({label_results[conference][year]['academia_open_source_code']:.2f}) than industry ({label_results[conference][year]['industry_open_source_code']:.2f})")
            academia_greater_count += 1
            academia_greater_by_year[year] += 1
        elif label_results[conference][year]["academia_open_source_code"] < label_results[conference][year]["industry_open_source_code"]:
            #print(f"In {conference} {year}, industry had more open source code ({label_results[conference][year]['industry_open_source_code']:.2f}) than academia ({label_results[conference][year]['academia_open_source_code']:.2f})")
            industry_greater_count += 1
            industry_greater_by_year[year] += 1
        else:
            print(f"In {conference} {year}, open source code was tied ({label_results[conference][year]['academia_open_source_code']:.2f})")
            tie_count += 1
        
        # compare academia and industry open datasets
        if label_results[conference][year]["academia_open_datasets"] > label_results[conference][year]["industry_open_datasets"]:
            #print(f"In {conference} {year}, academia had more open datasets ({label_results[conference][year]['academia_open_datasets']:.2f}) than industry ({label_results[conference][year]['industry_open_datasets']:.2f})")
            academia_greater_count += 1
            academia_greater_by_year[year] += 1
        elif label_results[conference][year]["academia_open_datasets"] < label_results[conference][year]["industry_open_datasets"]:
            #print(f"In {conference} {year}, industry had more open datasets ({label_results[conference][year]['industry_open_datasets']:.2f}) than academia ({label_results[conference][year]['academia_open_datasets']:.2f})")
            industry_greater_count += 1
            industry_greater_by_year[year] += 1
        else:
            print(f"In {conference} {year}, open datasets was tied ({label_results[conference][year]['academia_open_datasets']:.2f})")
            tie_count += 1
        
        # compare academia and industry dataset splits
        if label_results[conference][year]["academia_dataset_splits"] > label_results[conference][year]["industry_dataset_splits"]:
            #print(f"In {conference} {year}, academia had more dataset splits ({label_results[conference][year]['academia_dataset_splits']:.2f}) than industry ({label_results[conference][year]['industry_dataset_splits']:.2f})")
            academia_greater_count += 1
            academia_greater_by_year[year] += 1
        elif label_results[conference][year]["academia_dataset_splits"] < label_results[conference][year]["industry_dataset_splits"]:
            #print(f"In {conference} {year}, industry had more dataset splits ({label_results[conference][year]['industry_dataset_splits']:.2f}) than academia ({label_results[conference][year]['academia_dataset_splits']:.2f})")
            industry_greater_count += 1
            industry_greater_by_year[year] += 1
        else:
            print(f"In {conference} {year}, dataset splits was tied ({label_results[conference][year]['academia_dataset_splits']:.2f})")
            tie_count += 1

        # compare academia and industry hardware specification
        if label_results[conference][year]["academia_hardware_specification"] > label_results[conference][year]["industry_hardware_specification"]:
            #print(f"In {conference} {year}, academia had more hardware specification ({label_results[conference][year]['academia_hardware_specification']:.2f}) than industry ({label_results[conference][year]['industry_hardware_specification']:.2f})")
            academia_greater_count += 1
            academia_greater_by_year[year] += 1
        elif label_results[conference][year]["academia_hardware_specification"] < label_results[conference][year]["industry_hardware_specification"]:
            #print(f"In {conference} {year}, industry had more hardware specification ({label_results[conference][year]['industry_hardware_specification']:.2f}) than academia ({label_results[conference][year]['academia_hardware_specification']:.2f})")
            industry_greater_count += 1
            industry_greater_by_year[year] += 1
        else:
            print(f"In {conference} {year}, hardware specification was tied ({label_results[conference][year]['academia_hardware_specification']:.2f})")
            tie_count += 1
        
        # compare academia and industry software dependencies
        if label_results[conference][year]["academia_software_dependencies"] > label_results[conference][year]["industry_software_dependencies"]:
            #print(f"In {conference} {year}, academia had more software dependencies ({label_results[conference][year]['academia_software_dependencies']:.2f}) than industry ({label_results[conference][year]['industry_software_dependencies']:.2f})")
            academia_greater_count += 1
            academia_greater_by_year[year] += 1
        elif label_results[conference][year]["academia_software_dependencies"] < label_results[conference][year]["industry_software_dependencies"]:
            #print(f"In {conference} {year}, industry had more software dependencies ({label_results[conference][year]['industry_software_dependencies']:.2f}) than academia ({label_results[conference][year]['academia_software_dependencies']:.2f})")
            industry_greater_count += 1
            industry_greater_by_year[year] += 1
        else:
            print(f"In {conference} {year}, software dependencies was tied ({label_results[conference][year]['academia_software_dependencies']:.2f})")
            tie_count += 1

        # compare academia and industry experiment setup
        if label_results[conference][year]["academia_experiment_setup"] > label_results[conference][year]["industry_experiment_setup"]:
            #print(f"In {conference} {year}, academia had more experiment setup ({label_results[conference][year]['academia_experiment_setup']:.2f}) than industry ({label_results[conference][year]['industry_experiment_setup']:.2f})")
            academia_greater_count += 1
            academia_greater_by_year[year] += 1
        elif label_results[conference][year]["academia_experiment_setup"] < label_results[conference][year]["industry_experiment_setup"]:
            #print(f"In {conference} {year}, industry had more experiment setup ({label_results[conference][year]['industry_experiment_setup']:.2f}) than academia ({label_results[conference][year]['academia_experiment_setup']:.2f})")
            industry_greater_count += 1
            industry_greater_by_year[year] += 1
        else:
            print(f"In {conference} {year}, experiment setup was tied ({label_results[conference][year]['academia_experiment_setup']:.2f})")
            tie_count += 1


print(f"Academia had greater counts in {academia_greater_count} years."
      f" Industry had greater counts in {industry_greater_count} years."
      f" There were {tie_count} ties.")




In [ ]:
# Perform a binomial test
from scipy.stats import binomtest
n = academia_greater_count + industry_greater_count + tie_count
k = academia_greater_count
binom_result = binomtest(k, n, p=0.5, alternative='greater')
print(binom_result.pvalue)

k = industry_greater_count
binom_result = binomtest(k, n, p=0.5, alternative='greater')
print(binom_result.pvalue)

In [ ]:
# Create a datatable with years as columns, labels as rows, and 'A'/'I' for academia/industry greater
import pandas as pd

# Define the seven labels to compare
compare_labels = [
    "pseudocode",
    "open_source_code",
    "open_datasets",
    "dataset_splits",
    "hardware_specification",
    "software_dependencies",
    "experiment_setup"
]

# Prepare the table for a single conference (e.g., AAAI)
conference = "NeurIPS"  # Change as needed
years = sorted(label_results[conference].keys())
rows = []
for label in compare_labels:
    row = []
    academia_key = f"academia_{label}"
    industry_key = f"industry_{label}"
    for year in years:
        academia_val = label_results[conference][year][academia_key]
        industry_val = label_results[conference][year][industry_key]
        if academia_val > industry_val:
            row.append("A")
        elif industry_val > academia_val:
            row.append("I")
        else:
            row.append("")
    rows.append(row)

# Create DataFrame
datatable = pd.DataFrame(rows, index=compare_labels, columns=years)
datatable

In [ ]:
font_size = 18

# Line chart of academia_greater_by_year and industry_greater_by_year
import matplotlib.pyplot as plt

years = sorted(academia_greater_by_year.keys())
academia_counts = [academia_greater_by_year[year] for year in years]
industry_counts = [industry_greater_by_year[year] for year in years]

plt.figure(figsize=(6, 6))
plt.plot(years, academia_counts, label='Academia')
plt.plot(years, industry_counts, label='Industry')
plt.xlabel('Year', fontsize=font_size - 2)
plt.ylabel('Count (Across All Conferences and Reproducibility Variables)', fontsize=font_size - 2)
plt.ylabel('Count', fontsize=font_size - 2)

#plt.title('Number of Times Academia or Industry Had Greater Percentage\nof Documented Reproducibility Variables by Year', fontsize=font_size)
plt.title('Academia vs Industry Documentation Trends', fontsize=font_size)

# Show every year on x-axis
#plt.xticks(years)

# Show y-axis ticks and labels on both left and right
plt.gca().yaxis.set_ticks_position('both')
plt.gca().tick_params(axis='y', labelright=True)

# Set the font size of tick labels
plt.xticks(fontsize=font_size - 4)
plt.yticks(fontsize=font_size - 4)

# Show the legend on the top left
plt.legend(loc='upper left', ncol=1, fontsize=font_size - 4)
# Show only the grid lines for y-axis
plt.grid(axis='y')

plt.savefig("../../plots/academia_industry_reproducibility_variables.pdf", dpi=600, bbox_inches="tight")
plt.show()